# Author: Victor Oluwabiyi
## Title: Axis A: Text Representation
**Depends on:** `../Data/cleaned_full_dataset.csv` — produced by Talha Khan

## What this notebook does
This notebook takes the cleaned arXiv cs.LG abstracts and converts them into three different numerical representations for downstream analysis by Chenyu Yang (comparison methods) and Sahil Bhinde (PCA and evaluation).

## Representations covered
1. **TF-IDF** — sparse matrix based on term frequency, used as the baseline.
2. **Word2Vec** — dense embeddings trained on the corpus using gensim.
3. **Sentence-BERT** — dense embeddings from a pre-trained transformer model.

## Files produced
| File | Location | To be used by |
|------|----------|---------|
| `tfidf_matrix.npz` | `../Data/` | Chenyu Yang, Sahil Bhinde |
| `tfidf_vocab.pkl` | `../Data/` | Chenyu Yang |
| `word2vec_embeddings.npy` | `../Data/` | Chenyu Yang, Sahil Bhinde |
| `sbert_embeddings.npy` | `../Data/` | Chenyu Yang, Sahil Bhinde |

# Loading The Dataset
First of all, all the libraries needed to run the notebook are imported. The cleaned dataset containing only **cs.LG** (Machine Learning) data is loaded and inspected to make sure everything is in order

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from gensim.utils import tokenize
from sentence_transformers import SentenceTransformer

In [2]:
#Load the dataset from the Data folder
df = pd.read_csv("../Data/cleaned_full_dataset.csv")

#Inspect the dataset to see if it loaded properly
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(list(df.columns))
print(f"\nFirst 3 rows:")
print(df.head(3))
print(df["period"].value_counts())

#store cleaned_abstract, period and year for later use throughout this notebook
docs = df["cleaned_abstract"] # swap to complete cleaned_abstract when Person A delivers cleaned version
periods = df["period"]
years = df["year"]

print(f"\ndocs: {len(docs)} abstracts loaded")
print(f"periods: {periods.nunique()} unique periods")
print(f"years: {years.min()} - {years.max()}")

Shape: 7014 rows × 9 columns
['id', 'title', 'abstract', 'categories', 'year', 'cleaned_abstract', 'word_count', 'char_count', 'period']

First 3 rows:
          id                                              title  \
0  0704.1274   Parametric Learning and Monte Carlo Optimization   
1  0704.2668  Supervised Feature Selection via Dependence Es...   
2  0705.1585  HMM Speaker Identification Using Linear and No...   

                                            abstract categories  year  \
0  This paper uncovers and explores the close rel...  ['cs.LG']  2007   
1  We introduce a framework for filtering feature...  ['cs.LG']  2007   
2  Speaker identification is a powerful, non-inva...  ['cs.LG']  2007   

                                    cleaned_abstract  word_count  char_count  \
0  paper uncovers explores close relationship mon...         197        1340   
1  introduce framework filtering features employs...          76         542   
2  speaker identification powerful noninvasive

# Hypotheses for each Text Representation Approach

As mentioned above, these approaches below will be used to explore the text representation axis for the chosen dataset:

- TF-IDF
- Word2Vec
- Sentence-BERT

It is expected that the result of each approach will match the following hypotheses:
1. **TF-IDF:** It is expected that TF-IDF will focus more on keywords that distinguish one abstract from the others. Placing less emphasis on common words across different documents. Therefore I expect TD-IDF to clearly show a shift in vocabulary between periods, with modern terms appearing predominantly in later periods.
2. **Word2Vec:** It is expected that, unlike TF-IDF, Word2Vec will capture semantic relationships between terms, grouping similar concepts(e.g. synonyms) together even when different words are used. However since it is trained only on the arXiv corpus it may struggle with very rare or domain-specific terms that do not come up frequently. Therefore I expect Word2Vec to show clusters of similar methods or concepts from different periods even with the change in vocabulary.
3. **Sentence-BERT:** Since it is pre-trained on large general text rather than the corpus, it is expected that Sentence-BERT will produce the most semantically rich representations of the three approaches. Even with this advantage, it may underperform with highly technical and niche arXiv terminology that do not frequently appear in the pre-training data. Therefore I expect Sentence-BERT to produce a more gradual and consistent shift when abstracts are plotted over time, since it captures the overall meaning of a sentence(or abstract) rather than being thrown off by individual, similar words/terms changing between periods.